# Completion across February and October presentations

The outcome is Pass or Distinction among all enrollments. For the main factorial comparison, keep modules offered in both seasons in both years. The same module mix can then support both seasonal contrasts, the two same-season year comparisons and their interaction. The latter measures a change in association across years. There is no assigned intervention or untreated control.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

RAW, ASSETS = Path("../data/raw"), Path("../assets")
SEED = 42
KEYS = ["id_student", "code_module", "code_presentation"]
FOCAL = ("BBB", "2013B", "2014J")
students = pd.read_csv(RAW / "studentInfo.csv")
assert not students.duplicated(KEYS).any()
students["completed"] = students.final_result.isin(["Pass", "Distinction"]).astype(int)
students["j"] = students.code_presentation.str.endswith("J").astype(int)
students["year14"] = students.code_presentation.str.startswith("2014").astype(int)
students["imd_band"] = students.imd_band.fillna("Missing")
COV = ("C(age_band) + C(highest_education) + C(imd_band) + C(gender) + "
       "C(disability) + studied_credits + C(region) + num_of_prev_attempts")
support = pd.crosstab(students.code_module, students.code_presentation)
print("Enrolled by module and intake")
print(support.to_string())
common = support.index[support.gt(0).all(axis=1)]
panel = students.loc[students.code_module.isin(common)].copy()
print(panel.groupby("code_presentation").completed.agg(n="size", completed="sum", rate="mean").round(4))

def fit_model(data, formula):
    model = smf.ols("completed ~ " + formula, data, missing="raise")
    assert np.linalg.matrix_rank(model.exog) == model.exog.shape[1]
    return model.fit(cov_type="cluster", cov_kwds={"groups": data.id_student}, use_t=False)

def contrast(fit, weights):
    vector = pd.Series(0., index=fit.params.index)
    for term, weight in weights.items():
        vector[term] = weight
    est = fit.t_test(vector)
    lo, hi = est.conf_int().ravel()
    return {"pp": 100*est.effect.item(), "lo": 100*lo, "hi": 100*hi, "p": est.pvalue.item()}

def p_text(p_value):
    return '<0.0001' if p_value < 0.0001 else f'{p_value:.4f}'

## Common-support comparisons

Fit a linear probability model with a season-by-year interaction, module fixed effects and entry covariates. Its contrasts give percentage-point differences. The model assumes additive covariate and module effects, with a common season-by-year pattern across modules. Unadjusted contrasts use the same eligible enrollments. All intervals use student-cluster sandwich covariance and nominal two-sided 95% coverage. They do not describe uncertainty across universities or independent years.

In [ ]:
terms = {
    "2013 J minus B": {"j": 1},
    "2014 J minus B": {"j": 1, "j:year14": 1},
    "B: 2014 minus 2013": {"year14": 1},
    "J: 2014 minus 2013": {"year14": 1, "j:year14": 1},
    "Change in J minus B": {"j:year14": 1},
}
fits = {"raw": fit_model(panel, "j * year14"),
        "adjusted": fit_model(panel, "j * year14 + C(code_module) + " + COV)}
contrasts = pd.DataFrame([{"model": name, "comparison": label, **contrast(fit, weights)}
                          for name, fit in fits.items() for label, weights in terms.items()])
print(contrasts.round(4).to_string(index=False))

## Within-module estimates and the original BBB pair

Use available within-year B/J pairs, including modules outside the balanced panel. Report missing support explicitly. Each estimable module-year has its own covariate-adjusted regression. Multiple comparisons make individual intervals exploratory. For the original BBB cross-year contrast, retain the earlier covariate list and standardized logistic risk difference so its historical estimate can be reproduced.

In [ ]:
rows = []
for module, counts in support.iterrows():
    for year in ["2013", "2014"]:
        b, j = year + "B", year + "J"
        row = dict(module=module, year=year, B=counts[b], J=counts[j])
        if min(counts[b], counts[j]) > 0:
            part = students.loc[students.code_module.eq(module) & students.code_presentation.isin([b, j])]
            row.update(contrast(fit_model(part, "j + " + COV), {"j": 1}))
        rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False, na_rep="no pair"))
module, b, j = FOCAL
focal = students.loc[students.code_module.eq(module) & students.code_presentation.isin([b,j])].copy()
focal["j_presentation"] = focal.j
prior_cov = COV.rsplit(" + ", 1)[0]
fit = smf.glm("completed ~ j + " + prior_cov, focal, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": focal.id_student})
# Standardize both predictions over the same enrollment mix; delta-method cluster SE.
from patsy import build_design_matrices
from scipy.special import expit
def standardized(flag):
    design = build_design_matrices([fit.model.data.design_info], focal.assign(j=flag))[0]
    prob = expit(design @ fit.params.to_numpy())
    return prob.mean(), np.asarray(design).T @ (prob*(1-prob)) / len(prob)
p1, g1 = standardized(1)
p0, g0 = standardized(0)
rd = p1-p0
se = np.sqrt((g1-g0) @ fit.cov_params() @ (g1-g0))
print("BBB observed:", focal.groupby("code_presentation").completed.mean().round(4).to_dict())
print(f"BBB standardized J minus B: {100*rd:+.2f} pp [{100*(rd-1.96*se):+.2f}, "
      f"{100*(rd+1.96*se):+.2f}], p={2*stats.norm.sf(abs(rd/se)):.4f}")

## Sensitivity to an unmeasured confounder

For each within-year contrast on the common modules, estimate an adjusted completion risk ratio by modified Poisson regression. Student-cluster sandwich intervals allow binary outcomes. The E-value expresses the minimum equal risk-ratio associations a single confounder would need with both season and completion, conditional on measured covariates, to explain the estimate. The bound uses the confidence limit closest to one and equals one if the interval includes one. This addresses a specific confounding scenario, leaving selection and calendar changes unresolved.

In [ ]:
def evalue(rr):
    strength = max(rr, 1/rr)
    return strength + np.sqrt(strength*(strength-1))

rows = []
for year, part in panel.groupby("year14"):
    fit_rr = smf.glm("completed ~ j + C(code_module) + " + COV, part,
                     family=sm.families.Poisson(), missing="raise").fit(
                         cov_type="cluster", cov_kwds={"groups": part.id_student})
    rr = np.exp(fit_rr.params["j"])
    lo, hi = np.exp(fit_rr.conf_int().loc["j"])
    bound = np.clip(1., lo, hi)
    rows.append(dict(year=2013+year, RR=rr, lo=lo, hi=hi, E=evalue(rr), E_limit=evalue(bound)))
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
balance_cols = ['age_band', 'highest_education', 'imd_band', 'gender', 'disability', 'region']
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained')
for field, ax in zip(balance_cols, axes.flat):
    shares = pd.crosstab(focal[field], focal['code_presentation'], normalize='columns')
    shares.mul(100).plot.barh(ax=ax, legend=False)
    ax.set(title=field.replace('_', ' ').title(), ylabel='', xlabel='% of presentation')
axes[0, 0].legend(title='Presentation')
fig.suptitle('BBB enrollment composition by presentation')
fig.savefig(ASSETS / '03_group_balance.png', dpi=125, bbox_inches='tight')
plt.show()

## Secondary outcome — recorded assessment scores

Assessment IDs are joined to their own module and presentation. Each scored
enrollment contributes its mean across available assessment scores, including
recorded banked scores. Enrollment coverage is shown by presentation. Restricting
the outcome to scored enrollments creates selection (survivorship) bias. Tasks,
marking, and the mix of banked scores can also differ by presentation.

Welch's test compares means. Mann–Whitney U compares score ranks. Both are
conventional unpaired tests and treat enrollments as independent. The bootstrap
resamples student IDs across both presentations together, preserving repeated
students in the interval for the mean difference.

In [ ]:
assessments = pd.read_csv(RAW / 'assessments.csv', usecols=['id_assessment', *KEYS[1:]])
submissions = pd.read_csv(RAW / 'studentAssessment.csv', usecols=['id_assessment', 'id_student', 'score'])
student_scores = (
    submissions.dropna(subset=['score'])
    .merge(assessments, on='id_assessment', validate='many_to_one')
    .groupby(KEYS, as_index=False)['score'].mean()
)
scored = focal.merge(student_scores, on=KEYS, how='left', validate='one_to_one')
score_summary = scored.groupby('code_presentation')['score'].agg(
    enrolled='size', scored='count', mean='mean', median='median', sd='std'
)
score_summary['coverage (%)'] = 100 * score_summary['scored'] / score_summary['enrolled']
display(score_summary.round(2))

_, b, j = FOCAL
scores = scored.pivot(index='id_student', columns='code_presentation', values='score')[[b, j]].dropna(how='all')
left = scores[b].dropna().to_numpy()
right = scores[j].dropna().to_numpy()
welch = stats.ttest_ind(right, left, equal_var=False)
mann_whitney = stats.mannwhitneyu(right, left, alternative='two-sided')
delta = right.mean() - left.mean()
pooled_sd = np.sqrt(((len(left) - 1) * left.var(ddof=1) + (len(right) - 1) * right.var(ddof=1))
                    / (len(left) + len(right) - 2))
print(f'Welch t = {welch.statistic:.3f}, p {p_text(welch.pvalue)}')
print(f'Mann–Whitney U (J vs B) = {mann_whitney.statistic:,.0f}, p {p_text(mann_whitney.pvalue)}')
print(f'Score difference (J − B): {delta:+.2f} points. Cohen’s d: {delta / pooled_sd:+.3f}.')
print(f'Student IDs with a score in both presentations: {scores.notna().all(axis=1).sum():,}')

fig, (hist_ax, box_ax) = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
for label in (b, j):
    hist_ax.hist(scores[label].dropna(), bins=np.arange(0, 105, 5), alpha=0.55,
                 density=True, label=label)
hist_ax.set(xlabel='Mean recorded score', ylabel='Density')
hist_ax.legend()
box_ax.boxplot([left, right], tick_labels=[b, j])
box_ax.set(ylabel='Mean recorded score', ylim=(-2, 102))
fig.suptitle('Score distributions among scored BBB enrollments')
fig.savefig(ASSETS / '03_score_distribution.png', dpi=125, bbox_inches='tight')
plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
bootstrap_deltas = []
for _ in range(100):
    sampled = rng.integers(len(scores), size=(100, len(scores)))
    boot_means = np.nanmean(scores.to_numpy()[sampled], axis=1)
    bootstrap_deltas.extend(boot_means[:, 1] - boot_means[:, 0])

low, high = np.quantile(bootstrap_deltas, [0.025, 0.975])
print(f'Student-cluster bootstrap ({len(bootstrap_deltas):,} draws), 95% CI: [{low:+.2f}, {high:+.2f}] points.')
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
ax.hist(bootstrap_deltas, bins=35, color='steelblue')
ax.axvspan(low, high, color='orange', alpha=0.2, label='95% interval')
ax.axvline(delta, color='black', label='Observed difference')
ax.set(xlabel='Score difference (J − B)', ylabel='Bootstrap draws')
ax.legend()
fig.savefig(ASSETS / '03_bootstrap_ci.png', dpi=125, bbox_inches='tight')
plt.show()

## Sample size for a future randomized study

Illustrative planning assumes independent observations, equal assignment and
a baseline equal to the earlier BBB completion rate. The plot does not account
for clustering or multiple outcomes.

In [ ]:
baseline = focal.loc[focal['j_presentation'].eq(0), 'completed'].mean()
changes = np.arange(1, 11)
samples = [
    np.ceil(NormalIndPower().solve_power(
        proportion_effectsize(baseline + change / 100, baseline),
        power=0.8, alpha=0.05, ratio=1,
    ))
    for change in changes
]
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
ax.plot(changes, samples, marker='o')
ax.set(xlabel='Absolute completion difference (pp)', ylabel='Enrollments per arm', yscale='log')
fig.savefig(ASSETS / '03_power_analysis.png', dpi=125, bbox_inches='tight')
plt.show()
print(f'Illustrative sample per arm for a 5 pp difference: {samples[4]:,.0f}.')